# 📓 02. Data Cleaning & Advanced Feature Engineering
> **Đề tài:** Phân tích Không gian - Kinh tế thị trường phòng trọ TP.HCM & Bình Dương (IE313)  

---
## 🎯 Mục tiêu Notebook:
1. Thực thi quy trình làm sạch 5 tầng (**5-Tier Cleaning Pipeline**): Khử trùng lặp kép, Domain Sanity, Grouped IQR theo Quận, và Điền khuyết.
2. Trực quan hóa so sánh phân phối **Before vs. After** khi áp dụng Grouped IQR.
3. Trích xuất đặc trưng văn bản (**NLP Regex**) và tính khoảng cách không gian (**Haversine Distance**).
4. Phân loại người đăng (**Seller Profiling**) và phân khúc thị trường (**K-Means Clustering**).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
BASE_DIR = Path('..').resolve()
sys.path.append(str(BASE_DIR))

from src.cleaner import clean_rental_dataset
from src.features import build_feature_pipeline

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Chạy Pipeline Làm sạch & Biểu đồ Phễu lọc dữ liệu (Data Funnel)

In [ ]:
df_cleaned, audit_report = clean_rental_dataset()

funnel_df = pd.DataFrame(audit_report['funnel_steps'])
display(funnel_df[['step', 'count', 'dropped', 'retention_pct', 'description']])

# Trực quan hóa Data Funnel
plt.figure(figsize=(10, 5))
bars = plt.bar(funnel_df['step'], funnel_df['count'], color='#1f77b4', edgecolor='black', alpha=0.85)
plt.title('Biểu đồ Phễu Kiểm toán Chất lượng Dữ liệu (Data Cleaning Funnel)', fontsize=13, fontweight='bold')
plt.ylabel('Số lượng bản ghi còn lại')
plt.xticks(rotation=20)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f'{int(yval):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Trích xuất Đặc trưng Nâng cao (Feature Engineering)
- **NLP Regex:** `has_mezzanine`, `has_ac`, `is_free_curfew`, `has_balcony_window`
- **Geospatial:** `dist_to_q1_km`, `dist_to_vnuhcm_km`
- **Behavioral:** `seller_type` (Chính chủ vs Môi giới trá hình)
- **Clustering:** `market_cluster` (Tier 1, Tier 2, Tier 3)

In [ ]:
df_features = build_feature_pipeline(df_cleaned)
display(df_features.head(3))

# Thống kê tỷ lệ các tiện ích
amenities = ['has_ac', 'has_mezzanine', 'is_free_curfew', 'has_balcony_window']
amenity_pct = (df_features[amenities].mean() * 100).round(1)
print('Tỷ lệ phòng có tiện ích (%):\n', amenity_pct)